<a href="https://colab.research.google.com/github/snehubhosle/CSI_Weekly_Assignments/blob/main/Week6_Spark_Assignment/Notebook/Week6_Spark_Assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **Week-6 Assignment : Apache Spark**


### **Apache Spark Architecture, Data Processing & Performance Optimization using PySpark**

**Dataset: S**ample Superstore Dataset

**Author:** Snehal A. Bhosale

**College:** Sanjivani College of Engineering, Kopargaon – 423603

**E-mail:** snehalbhosale1807@gmail.com

**Technology Used:**

Apache Spark (PySpark)
Python
Google Colab / Jupyter Notebook
Apache Spark SQL
CSV & Parquet File Formats

### **Objective:**

To understand Apache Spark Architecture and perform efficient data processing using DataFrames, transformations, filtering, schema handling, lazy evaluation, and optimized storage formats (CSV & Parquet). The assignment also focuses on Spark performance concepts such as DAG, Predicate Pushdown, Shuffle operations, and best practices for handling large datasets.

### **Implementation:**

### **Step 1: Install Required Libraries**

In [1]:
!pip install pyspark

### **Step 2: Import Required Libraries**

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

### **Step 3: Create Spark Session**

In [3]:
spark = SparkSession.builder \
    .appName("Week6 Spark Assignment") \
    .getOrCreate()

Verify Spark

In [4]:
spark

### **Step 4: upload the CSV**

In [6]:
from google.colab import files
uploaded = files.upload()

Saving Sample - Superstore.csv to Sample - Superstore.csv


### **Step 5: Load the Dataset**

In [8]:
df = spark.read.csv(
    "Sample - Superstore.csv",
    header=True,
    inferSchema=True
)

### **Step 6: Explore the Dataset**

In [9]:
df.show(5)
df.printSchema()
df.count()

+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|  Customer Name|  Segment|      Country|           City|     State|Postal Code|Region|     Product ID|       Category|Sub-Category|        Product Name|   Sales|Quantity|Discount|  Profit|
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|     1|CA-2016-152156| 11/8/2016|11/11/2016|  Second Class|   CG-12520|    Claire Gute| Consumer|United States|      Henderson|  Kentucky|      42420| South|FUR-BO-10001798|      Furniture|   Bookcases|Bush Somerset 

9994

### **Step 7: Select and Filter Data**

In [19]:
from pyspark.sql.functions import col

df_processed.select("Product ID", "Category", "Sales").show()

df_processed.filter(col("Category") == "Technology").show()

df_processed.filter((col("Sales") > 1000) & (col("Region") == "West")).show()

df_processed.filter((col("Region") == "South") | (col("Category") == "Furniture")).show()

+---------------+---------------+--------+
|     Product ID|       Category|   Sales|
+---------------+---------------+--------+
|FUR-BO-10001798|      Furniture|  261.96|
|FUR-CH-10000454|      Furniture|  731.94|
|OFF-LA-10000240|Office Supplies|   14.62|
|FUR-TA-10000577|      Furniture|957.5775|
|OFF-ST-10000760|Office Supplies|  22.368|
|FUR-FU-10001487|      Furniture|   48.86|
|OFF-AR-10002833|Office Supplies|    7.28|
|TEC-PH-10002275|     Technology| 907.152|
|OFF-BI-10003910|Office Supplies|  18.504|
|OFF-AP-10002892|Office Supplies|   114.9|
|FUR-TA-10001539|      Furniture|1706.184|
|TEC-PH-10002033|     Technology| 911.424|
|OFF-PA-10002365|Office Supplies|  15.552|
|OFF-BI-10003656|Office Supplies| 407.976|
|OFF-AP-10002311|Office Supplies|   68.81|
|OFF-BI-10000756|Office Supplies|   2.544|
|OFF-ST-10004186|Office Supplies|  665.88|
|OFF-ST-10000107|Office Supplies|    55.5|
|OFF-AR-10003056|Office Supplies|    8.56|
|TEC-PH-10001949|     Technology|  213.48|
+----------

### **Step 8: Modify the DataFrame**

In [26]:
from pyspark.sql.functions import col, when
from pyspark.sql.types import DoubleType

# Helper function to safely cast to DoubleType, converting non-numeric strings to NULL
def safe_cast_to_double(column_name):
    # Ensure raw string for regex to avoid SyntaxWarning
    return when(col(column_name).cast("string").rlike(r"^-?\d*\.?\d+$"),
                col(column_name).cast(DoubleType())).otherwise(None)

# Rename Sales to Total_Sales
df = df.withColumnRenamed("Sales", "Total_Sales")

# Apply safe casting to Total_Sales, Quantity, and Discount
# This ensures that any malformed strings in these columns are converted to NULL instead of throwing an error.
df = df.withColumn("Total_Sales", safe_cast_to_double("Total_Sales"))
df = df.withColumn("Quantity", safe_cast_to_double("Quantity"))
df = df.withColumn("Discount", safe_cast_to_double("Discount"))

# Calculate Final_Price now that Total_Sales is correctly typed
df = df.withColumn("Final_Price", col("Total_Sales") * 1.18)

df.show(5)

+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+-----------+--------+--------+--------+------------------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|  Customer Name|  Segment|      Country|           City|     State|Postal Code|Region|     Product ID|       Category|Sub-Category|        Product Name|Total_Sales|Quantity|Discount|  Profit|       Final_Price|
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+-----------+--------+--------+--------+------------------+
|     1|CA-2016-152156| 11/8/2016|11/11/2016|  Second Class|   CG-12520|    Claire Gute| Consumer|United States|      Henderson|  Kentucky|      42420|

### **Step 9: Handle Missing Values**

In [38]:
from pyspark.sql.functions import col, when, count

df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df.columns
]).show()

df = df.na.drop()

+------+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+
|Row ID|Order ID|Order Date|Ship Date|Ship Mode|Customer ID|Customer Name|Segment|Country|City|State|Postal Code|Region|Product ID|Category|Sub-Category|Product Name|Sales|Quantity|Discount|Profit|
+------+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+
|     0|       0|         0|        0|        0|          0|            0|      0|      0|   0|    0|          0|     0|         0|       0|           0|           0|    0|       0|       0|     0|
+------+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+



### **Step 10: Perform Transformations and Actions**

In [29]:
# Transformations
filtered_df = df.filter(col("Region") == "West").select("Customer Name", "Total_Sales")

# Actions
filtered_df.show()
filtered_df.count()

+------------------+-----------+
|     Customer Name|Total_Sales|
+------------------+-----------+
|   Darrin Van Huff|      14.62|
|   Brosina Hoffman|      48.86|
|   Brosina Hoffman|       7.28|
|   Brosina Hoffman|    907.152|
|   Brosina Hoffman|     18.504|
|   Brosina Hoffman|      114.9|
|   Brosina Hoffman|   1706.184|
|   Brosina Hoffman|    911.424|
|      Irene Maddox|    407.976|
|   Alejandro Grove|       55.5|
|Zuschuss Donatelli|       8.56|
|Zuschuss Donatelli|     213.48|
|Zuschuss Donatelli|      22.72|
|       Emily Burns|    1044.63|
|     Eric Hoffmann|     11.648|
|     Eric Hoffmann|      90.57|
|      Ruben Ausman|      77.88|
|      Kunst Miller|      13.98|
|      Kunst Miller|     25.824|
|      Kunst Miller|     146.73|
+------------------+-----------+
only showing top 20 rows


3203

### **Step 11: Save Data in CSV and Parquet**

In [40]:
df.write.mode("overwrite").option("header", True).csv("output/csv_output")

df.write.mode("overwrite").parquet("output/parquet_output")

### **Step 12: Read Parquet and Demonstrate Predicate Pushdown**

In [41]:
parquet_df = spark.read.parquet("output/parquet_output")

parquet_df.filter(col("Region") == "West").show()

parquet_df.filter(col("Region") == "West").explain(True)

+------+--------------+----------+----------+--------------+-----------+------------------+---------+-------------+-------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|     Customer Name|  Segment|      Country|         City|     State|Postal Code|Region|     Product ID|       Category|Sub-Category|        Product Name|   Sales|Quantity|Discount|  Profit|
+------+--------------+----------+----------+--------------+-----------+------------------+---------+-------------+-------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|     3|CA-2016-138688| 6/12/2016| 6/16/2016|  Second Class|   DV-13045|   Darrin Van Huff|Corporate|United States|  Los Angeles|California|      90036|  West|OFF-LA-10000240|Office Supplies|      Labels|Self-Adhes

### **Step 13: Compare CSV and Parquet Performance**

In [46]:
# Read CSV
csv_df = spark.read.csv(
    "/content/Sample - Superstore.csv",
    header=True,
    inferSchema=True
)

# Read Parquet
parquet_df = spark.read.parquet("output/parquet_output")

# Compare number of records
print("CSV Records:", csv_df.count())
print("Parquet Records:", parquet_df.count())

# Display sample data
csv_df.show(5)
parquet_df.show(5)

CSV Records: 9994
Parquet Records: 9994
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|  Customer Name|  Segment|      Country|           City|     State|Postal Code|Region|     Product ID|       Category|Sub-Category|        Product Name|   Sales|Quantity|Discount|  Profit|
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|     1|CA-2016-152156| 11/8/2016|11/11/2016|  Second Class|   CG-12520|    Claire Gute| Consumer|United States|      Henderson|  Kentucky|      42420| South|FUR-BO-10001798|   

### **Step 14: Build the Spark Data Pipeline**

In [47]:
from pyspark.sql.functions import col

pipeline_df = (
    df
    .filter(col("Category") == "Technology")
    .withColumn("Final_Price", col("Sales") * 1.18)
)

pipeline_df.write.mode("overwrite").parquet("output/final_pipeline")

In [48]:
pipeline_df.printSchema()

root
 |-- Row ID: integer (nullable = true)
 |-- Order ID: string (nullable = true)
 |-- Order Date: string (nullable = true)
 |-- Ship Date: string (nullable = true)
 |-- Ship Mode: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Customer Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal Code: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Product Name: string (nullable = true)
 |-- Sales: double (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- Discount: double (nullable = true)
 |-- Profit: double (nullable = true)
 |-- Final_Price: double (nullable = true)



Verify the saved data


In [49]:
spark.read.parquet("output/final_pipeline").show(5)

+------+--------------+----------+----------+--------------+-----------+------------------+---------+-------------+-------------+----------+-----------+-------+---------------+----------+------------+--------------------+--------+--------+--------+--------+------------------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|     Customer Name|  Segment|      Country|         City|     State|Postal Code| Region|     Product ID|  Category|Sub-Category|        Product Name|   Sales|Quantity|Discount|  Profit|       Final_Price|
+------+--------------+----------+----------+--------------+-----------+------------------+---------+-------------+-------------+----------+-----------+-------+---------------+----------+------------+--------------------+--------+--------+--------+--------+------------------+
|     8|CA-2014-115812|  6/9/2014| 6/14/2014|Standard Class|   BH-11710|   Brosina Hoffman| Consumer|United States|  Los Angeles|California|      90032|   West|TEC-PH-10

### **Step 15: Performance Insights & Best Practices**

**Performance Insights**
1. Spark uses Lazy Evaluation, executing transformations only when an action (show(), count(), etc.) is called.  
2. Parquet files provide faster query performance and consume less storage than CSV due to their columnar format and compression.  
3. Predicate Pushdown filters data at the storage level, reducing the amount of data loaded into memory.  
4. Wide Transformations (e.g., groupBy()) involve data shuffling, while Narrow Transformations (e.g., filter(), select()) do not.  


**Best Practices**  
1. Use .show(5) for previewing data instead of .collect() on large datasets.
2. Define the schema or use inferSchema=True while reading data.
3. Prefer Parquet over CSV for storing processed datasets.
4. Filter data as early as possible to minimize processing.
5. Avoid unnecessary transformations and repeated actions to improve performance.

# **Week-6 : Spark Assignment Questions and Answers (Q1 to Q15)**



## **Q1. Explain the roles of the Driver, Cluster Manager, and Executor in a Spark application.**

## **Answer:**

Spark follows a **Master-Worker Architecture** where different components work together to execute distributed applications.

| Component           | Role                                                                                                                                      |
| ------------------- | ----------------------------------------------------------------------------------------------------------------------------------------- |
| **Driver**          | The main program that creates the Spark Session, converts code into tasks, schedules jobs, and collects results.                          |
| **Cluster Manager** | Allocates resources (CPU and memory) across the cluster and launches Executors. Examples include Standalone, YARN, Kubernetes, and Mesos. |
| **Executor**        | Worker processes running on cluster nodes. They execute tasks, store cached data, and send results back to the Driver.                    |

### **Execution Flow**

```
Driver Program
      │
      ▼
Cluster Manager
      │
      ▼
Executors
      │
      ▼
Process Data → Return Results
```

---



## **Q2. How does Spark's Lazy Evaluation strategy improve performance?**

## **Answer:**

Spark does **not execute transformations immediately**. Instead, it records them and creates a **Directed Acyclic Graph (DAG)**. Execution begins only when an **Action** (such as `show()` or `count()`) is called.

### **Advantages**

* Minimizes unnecessary computations.
* Optimizes execution using DAG.
* Reduces disk I/O.
* Combines multiple transformations into a single execution plan.

### **Example**




In [50]:

filtered_df = df.filter(col("Region") == "West") \
                .select("Customer Name", "Sales")

# No execution yet

filtered_df.show()      # Action triggers execution


+------------------+--------+
|     Customer Name|   Sales|
+------------------+--------+
|   Darrin Van Huff|   14.62|
|   Brosina Hoffman|   48.86|
|   Brosina Hoffman|    7.28|
|   Brosina Hoffman| 907.152|
|   Brosina Hoffman|  18.504|
|   Brosina Hoffman|   114.9|
|   Brosina Hoffman|1706.184|
|   Brosina Hoffman| 911.424|
|      Irene Maddox| 407.976|
|   Alejandro Grove|    55.5|
|Zuschuss Donatelli|    8.56|
|Zuschuss Donatelli|  213.48|
|Zuschuss Donatelli|   22.72|
|       Emily Burns| 1044.63|
|     Eric Hoffmann|  11.648|
|     Eric Hoffmann|   90.57|
|      Ruben Ausman|   77.88|
|      Kunst Miller|   13.98|
|      Kunst Miller|  25.824|
|      Kunst Miller|  146.73|
+------------------+--------+
only showing top 20 rows


# **Q3. Read a CSV file with Header and Infer Schema**

### **Answer**

* `header=True` → Uses the first row as column names.
* `inferSchema=True` → Automatically detects column data types.

---



In [52]:
df = spark.read.csv(
    "/content/Sample - Superstore.csv",
    header=True,
    inferSchema=True
)

# **Q4. Difference between CSV and Parquet**

### **Answer**

| Feature           | CSV            | Parquet          |
| ----------------- | -------------- | ---------------- |
| Storage Type      | Row-based      | Columnar         |
| Compression       | No             | Yes              |
| Schema            | Not Stored     | Stored           |
| Query Performance | Slower         | Faster           |
| Storage Size      | Larger         | Smaller          |
| Analytics         | Less Efficient | Highly Efficient |

### **Why Parquet is Faster?**

Parquet reads only the required columns, reducing disk I/O and improving query performance. It also supports **Predicate Pushdown**, making it ideal for big data analytics.

---



# **Q5. Select `product_id` and `price` where category is 'Electronics'**

### **Answer**



In [59]:
from pyspark.sql.functions import col

df.filter(col("Category") == "Technology") \
  .select("Product ID", "Sales") \
  .show()

+---------------+--------+
|     Product ID|   Sales|
+---------------+--------+
|TEC-PH-10002275| 907.152|
|TEC-PH-10002033| 911.424|
|TEC-PH-10001949|  213.48|
|TEC-AC-10003027|   90.57|
|TEC-PH-10004977|1097.544|
|TEC-PH-10000486| 371.168|
|TEC-PH-10004093| 147.168|
|TEC-AC-10000171|   45.98|
|TEC-AC-10002167|      45|
|TEC-PH-10003988|    21.8|
|TEC-PH-10002447| 1029.95|
|TEC-AC-10002167|      30|
|TEC-AC-10004633|   13.98|
|TEC-PH-10002726| 167.968|
|TEC-AC-10001998|   19.99|
|TEC-PH-10004093|  73.584|
|TEC-AC-10001767|  95.976|
|TEC-AC-10001552| 238.896|
|TEC-AC-10003499|  74.112|
|TEC-PH-10002844|  27.992|
+---------------+--------+
only showing top 20 rows


# **Q6. Rename a column and cast data type**

### **Answer**





In [61]:
from pyspark.sql.functions import col

# Rename Sales to Total_Sales
df = df.withColumnRenamed("Sales", "Total_Sales")

# Cast Total_Sales to Double
df = df.withColumn(
    "Total_Sales",
    col("Total_Sales").cast("double")
)

df.show(5)

+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+-----------+--------+--------+--------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|  Customer Name|  Segment|      Country|           City|     State|Postal Code|Region|     Product ID|       Category|Sub-Category|        Product Name|Total_Sales|Quantity|Discount|  Profit|
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+-----------+--------+--------+--------+
|     1|CA-2016-152156| 11/8/2016|11/11/2016|  Second Class|   CG-12520|    Claire Gute| Consumer|United States|      Henderson|  Kentucky|      42420| South|FUR-BO-10001798|      Furniture|   Bookcases|Bush 

# **Q7. How does Spark's DAG provide Fault Tolerance?**

### **Answer**

Spark maintains a **Lineage Graph (DAG)** that records every transformation applied to the data.

If an Executor or worker node fails:

* Spark identifies the lost partition.
* Recomputes only the missing partition using the lineage graph.
* Does **not** recompute the entire dataset.

### **Benefits**

* Automatic recovery.
* No data replication required.
* Efficient fault recovery.

---



# **Q8. Filter orders where Status is 'Completed' and Amount > 1000**

### **Answer**


In [92]:
from pyspark.sql.functions import col, when
from pyspark.sql.types import DoubleType

# Helper function to safely cast to DoubleType, converting non-numeric strings to NULL
def safe_cast_to_double(column_name):
    # Ensure raw string for regex to avoid SyntaxWarning
    return when(col(column_name).cast("string").rlike(r"^-?\d*\.?\d+$"),
                col(column_name).cast(DoubleType())).otherwise(None)

# Apply safe casting to the 'Sales' column before filtering
df_fresh_processed = df_fresh.withColumn("Sales", safe_cast_to_double("Sales"))

df_fresh_processed.filter(col("Sales") > 1000).show()

+------+--------------+----------+----------+--------------+-----------+-----------------+-----------+-------------+-------------+------------+-----------+-------+---------------+---------------+------------+--------------------+--------+--------+--------+----------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|    Customer Name|    Segment|      Country|         City|       State|Postal Code| Region|     Product ID|       Category|Sub-Category|        Product Name|   Sales|Quantity|Discount|    Profit|
+------+--------------+----------+----------+--------------+-----------+-----------------+-----------+-------------+-------------+------------+-----------+-------+---------------+---------------+------------+--------------------+--------+--------+--------+----------+
|    11|CA-2014-115812|  6/9/2014| 6/14/2014|Standard Class|   BH-11710|  Brosina Hoffman|   Consumer|United States|  Los Angeles|  California|      90032|   West|FUR-TA-10001539|      Furniture| 

# **Q9. Explain Predicate Pushdown**

### **Answer**

**Predicate Pushdown** is an optimization technique used mainly with **Parquet** and other columnar storage formats.

Instead of loading the entire dataset into memory, Spark sends the filter condition directly to the storage layer. Only matching rows are read.

### **Benefits**

* Reads less data.
* Reduces memory usage.
* Improves query execution speed.
* Minimizes disk I/O.

### **Example**


In [93]:
parquet_df.filter(col("Region") == "West").show()

+------+--------------+----------+----------+--------------+-----------+------------------+---------+-------------+-------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|     Customer Name|  Segment|      Country|         City|     State|Postal Code|Region|     Product ID|       Category|Sub-Category|        Product Name|   Sales|Quantity|Discount|  Profit|
+------+--------------+----------+----------+--------------+-----------+------------------+---------+-------------+-------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|     3|CA-2016-138688| 6/12/2016| 6/16/2016|  Second Class|   DV-13045|   Darrin Van Huff|Corporate|United States|  Los Angeles|California|      90036|  West|OFF-LA-10000240|Office Supplies|      Labels|Self-Adhes

# **Q10. Add a new column `final_price`**

### **Answer**



In [96]:
from pyspark.sql.functions import col

df = df.withColumn(
    "final_price",
    col("Total_Sales") * 1.18
)

df.show()

+------+--------------+----------+----------+--------------+-----------+------------------+-----------+-------------+---------------+--------------+-----------+-------+---------------+---------------+------------+--------------------+-----------+--------+--------+--------+------------------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|     Customer Name|    Segment|      Country|           City|         State|Postal Code| Region|     Product ID|       Category|Sub-Category|        Product Name|Total_Sales|Quantity|Discount|  Profit|       final_price|
+------+--------------+----------+----------+--------------+-----------+------------------+-----------+-------------+---------------+--------------+-----------+-------+---------------+---------------+------------+--------------------+-----------+--------+--------+--------+------------------+
|     1|CA-2016-152156| 11/8/2016|11/11/2016|  Second Class|   CG-12520|       Claire Gute|   Consumer|United States|    

# **Q11. Difference between Transformations and Actions**

### **Answer**

| Transformations        | Actions                          |
| ---------------------- | -------------------------------- |
| Create a new DataFrame | Execute the computation          |
| Lazy Evaluation        | Trigger execution                |
| Return a DataFrame     | Return values or display results |

### **Examples**



In [98]:
#### Transformations

df.filter(col("Total_Sales") > 1000)

df.select("Customer Name", "Total_Sales")


#### Actions

df.show()

df.count()

+------+--------------+----------+----------+--------------+-----------+------------------+-----------+-------------+---------------+--------------+-----------+-------+---------------+---------------+------------+--------------------+-----------+--------+--------+--------+------------------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|     Customer Name|    Segment|      Country|           City|         State|Postal Code| Region|     Product ID|       Category|Sub-Category|        Product Name|Total_Sales|Quantity|Discount|  Profit|       final_price|
+------+--------------+----------+----------+--------------+-----------+------------------+-----------+-------------+---------------+--------------+-----------+-------+---------------+---------------+------------+--------------------+-----------+--------+--------+--------+------------------+
|     1|CA-2016-152156| 11/8/2016|11/11/2016|  Second Class|   CG-12520|       Claire Gute|   Consumer|United States|    

9994

# **Q12. Load Parquet → Filter Null Values → Save as CSV**

### **Answer**



In [103]:
from pyspark.sql.functions import col

parquet_df_read = spark.read.parquet("output/parquet_output")

parquet_df_read.filter(
    col("Sales").isNotNull()
).write.mode("overwrite") \
 .option("header", True) \
 .csv("output/filtered_parquet_to_csv")

# **Q13. Difference between Client Mode and Cluster Mode**

### **Answer**

| Client Mode                                       | Cluster Mode                                      |
| ------------------------------------------------- | ------------------------------------------------- |
| Driver runs on the local machine.                 | Driver runs inside the cluster.                   |
| Suitable for development and testing.             | Suitable for production environments.             |
| If the client disconnects, the application stops. | Continues running even if the client disconnects. |
| Easier debugging.                                 | Better scalability and fault tolerance.           |

---



# **Q14. Filter Region is 'North' OR Priority is 'High'**

### **Answer**

*Note:The Sample Superstore dataset does not contain a Priority column. Therefore, the query has been adapted to filter records based on the available Region column.*

In [107]:

df.filter(
    col("Region") == "North"
).show()


+------+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+------------+-----------+--------+--------+------+-----------+
|Row ID|Order ID|Order Date|Ship Date|Ship Mode|Customer ID|Customer Name|Segment|Country|City|State|Postal Code|Region|Product ID|Category|Sub-Category|Product Name|Total_Sales|Quantity|Discount|Profit|final_price|
+------+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+------------+-----------+--------+--------+------+-----------+
+------+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+------------+-----------+--------+--------+------+-----------+



# **Q15. Why use `.show(5)` instead of `.collect()` on large datasets?**

### **Answer**

The `.show(5)` function displays only the first **five rows** of a DataFrame, whereas `.collect()` retrieves **all rows** from every Executor to the Driver.

For very large datasets (GBs or TBs), using `.collect()` can:

* Consume excessive Driver memory.
* Cause **OutOfMemoryError**.
* Slow down application performance.
* Increase network traffic.





## **Key Learning Outcomes**

* Understood the **Apache Spark Architecture**, including the roles of the Driver, Cluster Manager, and Executors.
* Learned Spark's **Lazy Evaluation** and **DAG execution** for optimized processing.
* Performed DataFrame operations such as **reading, filtering, selecting, renaming, casting, and adding new columns**.
* Applied **data cleaning** techniques by handling null values and schema management.
* Differentiated between **Transformations** and **Actions** in Spark.
* Compared **CSV** and **Parquet** file formats and understood the performance benefits of Parquet.
* Explored **Predicate Pushdown**, **Shuffle**, and other Spark performance optimization concepts.
* Built an end-to-end **Spark data pipeline** for reading, transforming, filtering, and saving processed data.
* Followed Spark best practices for efficiently processing large datasets.


## **Conclusion**

This assignment provided practical experience with **Apache Spark Fundamentals** and **PySpark DataFrame operations**. It covered Spark architecture, data transformations, schema handling, performance optimization, and file formats. By implementing an end-to-end data pipeline, the assignment strengthened the understanding of scalable data processing and industry-standard Spark practices, forming a strong foundation for advanced data engineering and big data applications.
